# Aykırı Değer Baskılama Yöntemi (Winsorization/Capping)
Aykırı değerleri silmek ya da tek bir sayıyla değiştirmek yerine, IQR 
sınırlarının **kendisine "baskılamak"** — yani üst sınırdan büyük 
değerleri üst sınıra, alt sınırdan küçük değerleri alt sınıra çekmek.

```python
df['Deger_Baskilanmis'] = df['Deger'].clip(lower=alt_sinir, upper=ust_sinir)
```

**Avantajı:** Diğer iki yönteme göre **daha dengeli** — veri kaybı yok (silme gibi), ve her aykırı değer **kendi orijinal büyüklüğüne yakın** bir değere çekiliyor (ortalama atamadaki gibi hepsi aynı sayı olmuyor). Bu yüzden pratikte genellikle **en çok tercih edilen** yöntemdir.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
veri = np.random.normal(50, 10, 100)
veri = np.append(veri, [120, 130, -20])  # 3 bilerek eklenen aykırı değer
df = pd.DataFrame({'Deger': veri})

Q1 = df['Deger'].quantile(0.25)
Q3 = df['Deger'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR
df['Aykiri_IQR'] = (df['Deger'] < alt_sinir) | (df['Deger'] > ust_sinir)

# --- Yöntem 1: Silme ---
df_silme = df[~df['Aykiri_IQR']]

# --- Yöntem 2: Medyan ile Değiştirme ---
medyan_deger = df['Deger'].median()
df['Deger_Medyan'] = df['Deger'].where(~df['Aykiri_IQR'], medyan_deger)

# --- Yöntem 3: Baskılama (Winsorization) ---
df['Deger_Baskilanmis'] = df['Deger'].clip(lower=alt_sinir, upper=ust_sinir)

# --- Karşılaştırma ---
print("Orijinal veri    - ortalama:", round(df['Deger'].mean(),2), "std:", round(df['Deger'].std(), 2))
print("Silme sonrası    - ortalama:", round(df_silme['Deger'].mean(),2), "std:", round(df_silme['Deger'].std(),2))
print("Medyan atama     - ortalama:", round(df['Deger_Medyan'].mean(),2), "std:", round(df['Deger_Medyan'].std(),2))
print("Baskılama        - ortalama:", round(df['Deger_Baskilanmis'].mean(),2), "std:", round(df['Deger_Baskilanmis'].std(),2))

Orijinal veri    - ortalama: 49.77 std: 15.49
Silme sonrası    - ortalama: 49.22 std: 8.76
Medyan atama     - ortalama: 49.2 std: 8.59
Baskılama        - ortalama: 49.23 std: 9.64


### Sonuç
Silme, Medyan atama ve Baskılama yöntemlerini uyguladığımızda ortalama orijinal duruma (49.77)'e göre oldukça yakın kalırken (49.22, 49.20, 49.23), standart sapmalar aykırı değer silme sonrası ve aykırı değer medyan atama yöntemlerinde neredeyse yarıya düşerken, yani verinin varyansı azalırken, baskılama yönteminde standart sapma (9.64) diğer 2 yönteme göre biraz daha yüksek kalarak verinin varyansını biraz korumuş oluyoruz.